In [1]:
!pip install -q datasets soundfile transformers accelerate torch resemblyzer librosa pandas scipy

In [2]:
!pip install --upgrade sympy

In [3]:
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import os
from pathlib import Path
from scipy.signal import resample
from transformers import VitsModel, AutoTokenizer
from resemblyzer import VoiceEncoder, preprocess_wav
import glob
import zipfile
from google.colab import drive

# Config
NUM_SAMPLES_PER_EMOTION = 10
MODEL_NAME = "facebook/mms-tts-urd-script_arabic"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading models to {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = VitsModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
encoder = VoiceEncoder()

def run_evaluation(dataset_name, samples_list, text_key):
    base_dir = f"mushra_{dataset_name}"
    for sub in ["reference", "generated", "anchor"]:
        os.makedirs(f"{base_dir}/{sub}", exist_ok=True)

    results = []
    # Increased limit to 60 to accommodate 10 samples per 6 emotions
    for i in range(min(len(samples_list), 60)):
        try:
            sample = samples_list[i]
            text = sample[text_key]
            ref_array = np.array(sample['audio']['array']).astype(np.float32)
            ref_sr = sample['audio']['sampling_rate']

            if ref_array.ndim > 1:
                ref_array = ref_array.mean(axis=1)

            ref_path = f"{base_dir}/reference/sample_{i}.wav"
            gen_path = f"{base_dir}/generated/sample_{i}.wav"
            anc_path = f"{base_dir}/anchor/sample_{i}.wav"

            # Save Reference
            sf.write(ref_path, ref_array, ref_sr)

            # Generate TTS
            inputs = tokenizer(text, return_tensors="pt").to(device)
            with torch.no_grad():
                output = model(**inputs)
            gen_speech = output.waveform[0].cpu().numpy()
            gen_sr = model.config.sampling_rate
            sf.write(gen_path, gen_speech, gen_sr)

            # Create Anchor (8kHz cycle)
            target_len_8k = int(len(ref_array) * 8000 / ref_sr)
            downsampled = resample(ref_array, target_len_8k)
            upsampled = resample(downsampled, len(ref_array))
            sf.write(anc_path, upsampled, ref_sr)

            # Similarity
            wav_ref = preprocess_wav(Path(ref_path))
            wav_gen = preprocess_wav(Path(gen_path))
            emb_ref = encoder.embed_utterance(wav_ref)
            emb_gen = encoder.embed_utterance(wav_gen)
            sim = np.dot(emb_ref, emb_gen) / (np.linalg.norm(emb_ref) * np.linalg.norm(emb_gen))

            results.append({
                "sample_id": i,
                "emotion": sample.get("emotion", "unknown"),
                "urdu_text": text,
                "ref_gen_similarity": float(sim)
            })
            print(f"Sample {i+1} | Emotion: {sample.get('emotion','?')} | Similarity={sim:.4f}")

        except Exception as e:
            print(f"Error in sample {i}: {e}")
            continue

    return results

Loading models to cuda...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/302 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Loaded the voice encoder model on cuda in 0.14 seconds.


In [12]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/urduser.zip'
extract_to = '/content/UrduSER/'

if os.path.exists(zip_path):
    print(f"Unzipping {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("✅ Unzipped successfully!")
    # List contents to verify structure
    print("Folder contents:", os.listdir(extract_to))
else:
    print(f"❌ Could not find {zip_path} in your Drive root.")

Unzipping /content/drive/MyDrive/urduser.zip...
✅ Unzipped successfully!
Folder contents: ['UrduSER A Dataset for Urdu Speech Emotion Recognition']


In [16]:
import os

def list_all_files(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')

print('--- Full Directory Structure of /content/UrduSER ---')
list_all_files('/content/UrduSER/')

--- Full Directory Structure of /content/UrduSER ---
/
UrduSER A Dataset for Urdu Speech Emotion Recognition/
    UrduSER_Description.xlsx
    Sad/
        2_0_7_44.wav
        1_0_7_11.wav
        3_0_7_05.wav
        4_0_7_44.wav
        6_1_7_28.wav
        8_1_7_22.wav
        2_0_7_01.wav
        6_1_7_34.wav
        5_0_7_38.wav
        5_0_7_08.wav
        3_0_7_26.wav
        10_1_7_01.wav
        10_1_7_15.wav
        8_1_7_45.wav
        7_1_7_23.wav
        8_1_7_38.wav
        4_0_7_20.wav
        7_1_7_04.wav
        3_0_7_23.wav
        9_1_7_03.wav
        9_1_7_43.wav
        7_1_7_05.wav
        9_1_7_31.wav
        5_0_7_30.wav
        8_1_7_05.wav
        1_0_7_27.wav
        7_1_7_09.wav
        2_0_7_35.wav
        8_1_7_21.wav
        2_0_7_08.wav
        9_1_7_05.wav
        3_0_7_49.wav
        8_1_7_14.wav
        1_0_7_20.wav
        7_1_7_03.wav
        2_0_7_16.wav
        9_1_7_38.wav
        7_1_7_17.wav
        6_1_7_09.wav
        2_0_7_48.wav
        7_

In [21]:
# Re-running evaluation and ensuring results are saved
base_data_path = "/content/UrduSER/UrduSER A Dataset for Urdu Speech Emotion Recognition/"
metadata_path = os.path.join(base_data_path, "UrduSER_Description.xlsx")

try:
    df = pd.read_excel(metadata_path, header=1)
    file_col = "فائل کا نام"
    text_col = "تشریح"
    emotion_col = "جذبات"

    NEUTRAL_LABELS = {"neutral", "غیر جانبدار", ""}
    all_emotions = sorted([e for e in df[emotion_col].unique() if str(e).strip().lower() not in NEUTRAL_LABELS])

    urduser_samples = []
    for emotion in all_emotions:
        emotion_df = df[df[emotion_col] == emotion].head(NUM_SAMPLES_PER_EMOTION)
        for _, row in emotion_df.iterrows():
            file_name = str(row[file_col]).strip()
            audio_paths = glob.glob(f"{base_data_path}/**/{file_name}.wav", recursive=True)
            if audio_paths:
                ref_array, ref_sr = sf.read(audio_paths[0])
                if ref_array.ndim > 1: ref_array = np.mean(ref_array, axis=1)
                urduser_samples.append({
                    "audio": {"array": ref_array.astype(np.float32), "sampling_rate": int(ref_sr)},
                    "transcription": str(row[text_col]),
                    "emotion": str(emotion)
                })

    print(f"Loaded {len(urduser_samples)} samples. Starting evaluation...")
    results_urduser = run_evaluation("urduser", urduser_samples, text_key="transcription")

    # Explicitly create and save the DataFrame
    df_urduser = pd.DataFrame(results_urduser)
    if not df_urduser.empty:
        df_urduser.to_csv("urduser_evaluation_results_10per_emotion.csv", index=False)
        print(f"✅ CSV saved successfully with {len(df_urduser)} rows.")
        display(df_urduser.head())
    else:
        print("❌ Evaluation produced no results.")
except Exception as e:
    print(f"Error: {e}")

Loaded 60 samples. Starting evaluation...
Sample 1 | Emotion: افسردہ | Similarity=0.6071
Sample 2 | Emotion: افسردہ | Similarity=0.5030
Sample 3 | Emotion: افسردہ | Similarity=0.6036
Sample 4 | Emotion: افسردہ | Similarity=0.6502
Sample 5 | Emotion: افسردہ | Similarity=0.6029
Sample 6 | Emotion: افسردہ | Similarity=0.6504
Sample 7 | Emotion: افسردہ | Similarity=0.5836
Sample 8 | Emotion: افسردہ | Similarity=0.6224
Sample 9 | Emotion: افسردہ | Similarity=0.5564
Sample 10 | Emotion: افسردہ | Similarity=0.5508
Sample 11 | Emotion: بوریت | Similarity=0.6013
Sample 12 | Emotion: بوریت | Similarity=0.5953
Sample 13 | Emotion: بوریت | Similarity=0.5818
Sample 14 | Emotion: بوریت | Similarity=0.6570
Sample 15 | Emotion: بوریت | Similarity=0.6021
Sample 16 | Emotion: بوریت | Similarity=0.6527
Sample 17 | Emotion: بوریت | Similarity=0.5855
Sample 18 | Emotion: بوریت | Similarity=0.6694
Sample 19 | Emotion: بوریت | Similarity=0.4990
Sample 20 | Emotion: بوریت | Similarity=0.5491
Sample 21 | Emoti

,sample_id,emotion,urdu_text,ref_gen_similarity
0,0,افسردہ,بہت برا ہوں,0.607118
1,1,افسردہ,میں ساری زندگی جہاں ارا سے نفرت کرتا رہا,0.503023
2,2,افسردہ,اور اس کی سزا میں نے تمہیں بھی دی,0.603610
3,3,افسردہ,دل میں تھوڑی گنجائش پیدا کر لیتا تو تین زندگیا...,0.650185
4,4,افسردہ,میں جہاں ارا اور تم ہم اکٹھے رہتے,0.602874


In [22]:
import shutil
from google.colab import files

# Re-zipping the audio folders and downloading both zip and the populated CSV
shutil.make_archive("mushra_urduser_60samples_10per_emotion", 'zip', "mushra_urduser")
files.download("mushra_urduser_60samples_10per_emotion.zip")
files.download("urduser_evaluation_results_10per_emotion.csv")
print("✅ All files downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All files downloaded!
